env: Tensorflow (python 3.9.16)

In [9]:
%reset -f

In [ ]:
import numpy as np
import pickle

np.random.seed(10)

# -------------------------------
edges = [(1, 2), (2, 3), (1, 4), (2, 5), (3, 6),
         (4, 5), (5, 6), (4, 7), (5, 8), (6, 9),
         (7, 8), (8, 9)]
nodes = set(node for edge in edges for node in edge)
num_nodes = len(nodes)
terminals = [1, 9]

# -------------------------------
def sample_generator(dist_type, params):
    if dist_type == 'exponential':
        return np.random.exponential(1 / params['lambda'])
    elif dist_type == 'weibull':
        return np.random.weibull(params['b']) * params['a']
    elif dist_type == 'lognormal':
        return np.random.lognormal(params['mu'], params['sigma'])
    elif dist_type == 'infinite':   
        return np.inf
    else:
        raise ValueError(f"Unsupported distribution: {dist_type}")

# -------------------------------
node_distributions = {}
for n in nodes:
    if n in {1, 9}:  
        node_distributions[n] = ('infinite', {})
    if n in {3, 5, 7}:
        node_distributions[n] = ('exponential', {'lambda': 1.2})
    elif n in {2, 4, 6, 8}:
        node_distributions[n] = ('exponential', {'lambda': 0.7})

# -------------------------------
edge_distributions = {}
for idx, e in enumerate(edges, start=1):
    if idx in {1, 2, 6, 7, 11, 12}:
        edge_distributions[e] = ('weibull', {'a': 1.5, 'b': 3.6})
    elif idx in {3, 4, 5, 8, 9, 10}:
        edge_distributions[e] = ('lognormal', {'mu': 1.5, 'sigma': 2.6})

# -------------------------------
def generate_node_samples(distributions, n):
    samples_list = []
    for _ in range(n):
        sample = [(key, sample_generator(dist_type, params))
                  for key, (dist_type, params) in distributions.items()]
        samples_list.append(sample)
    return samples_list

def generate_edge_samples(edge_distributions, n):
    samples_list = []
    for _ in range(n):
        sample = [(u, v, sample_generator(dist_type, params))
                  for (u, v), (dist_type, params) in edge_distributions.items()]
        samples_list.append(sample)
    return samples_list

# -------------------------------
N_MCS = 15000

node_samples = generate_node_samples(node_distributions, N_MCS)
with open('data_lifetime_samples_node_' + str(N_MCS) +'.pkl', 'wb') as f:
    pickle.dump(node_samples, f)

edge_samples = generate_edge_samples(edge_distributions, N_MCS)
with open('data_lifetime_samples_edge_' + str(N_MCS) +'.pkl', 'wb') as f:
    pickle.dump(edge_samples, f)


In [11]:
node_samples[0], node_samples[1]

([(1, inf),
  (2, 2.1077634936374037),
  (3, 0.01747524758657736),
  (4, 1.434516143413345),
  (5, 1.151267750394458),
  (6, 0.9859509347644491),
  (7, 0.21219157598696198),
  (8, 0.31532151310706935),
  (9, inf)],
 [(1, inf),
  (2, 2.041900150007311),
  (3, 0.15438239196150588),
  (4, 0.13212565943232132),
  (5, 0.963604644657371),
  (6, 4.380017089324337),
  (7, 0.0032967344238430795),
  (8, 1.025477047297613),
  (9, inf)])

In [12]:
edge_samples[0], edge_samples[1]

([(1, 2, 0.8625886670606948),
  (2, 3, 1.0738153023593842),
  (1, 4, 6.776267248727984),
  (2, 5, 0.4792846495405082),
  (3, 6, 1.2276578337721507),
  (4, 5, 0.9961837175681924),
  (5, 6, 0.48089280868233264),
  (4, 7, 0.15614467914389152),
  (5, 8, 716.1300017857193),
  (6, 9, 0.4130821819292193),
  (7, 8, 0.8990995204390493),
  (8, 9, 1.4317526998507253)],
 [(1, 2, 1.4367112621419145),
  (2, 3, 1.672190916469491),
  (1, 4, 1.0029172136665534),
  (2, 5, 3.3362793616801105),
  (3, 6, 3.521034715559769),
  (4, 5, 1.6649401260492018),
  (5, 6, 2.2491120674729994),
  (4, 7, 1.0424166750798292),
  (5, 8, 0.1283618318945912),
  (6, 9, 244.92382135063974),
  (7, 8, 1.4652866980025312),
  (8, 9, 1.7627069850956356)])